In [ ]:
pip install tensorflow-datasets


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
import tensorflow_datasets as tfds

In [ ]:
LATENT_DIM = 32
IMAGE_SIZE = 64
IMAGE_SHAPE = (IMAGE_SIZE, IMAGE_SIZE, 3)

BATCH_SIZE = 128
EPOCHS = 30
LEARNING_RATE = 1e-4

In [ ]:
KL_WEIGHT = 1e-3

In [ ]:
def load_and_preprocess_faces_from_lfw():
    """
    Loads the LFW dataset from TFDS and returns train_ds, test_ds, x_test_images.
    Images are resized to IMAGE_SIZE x IMAGE_SIZE and normalized to [0, 1].
    """
    ds, ds_info = tfds.load(
        "lfw",
        split="train",
        with_info=True,
        as_supervised=False
    )

    num_examples = ds_info.splits["train"].num_examples
    train_size = int(0.9 * num_examples)
     # Split: first 90% train, remaining 10% test
    train_raw = ds.take(train_size)
    test_raw = ds.skip(train_size)

    def preprocess(sample):
        # sample["image"]: uint8 [H, W, 3]
        image = tf.cast(sample["image"], tf.float32) / 255.0
        image = tf.image.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
        return image

    AUTOTUNE = tf.data.AUTOTUNE

    train_ds = (
        train_raw
        .map(preprocess, num_parallel_calls=AUTOTUNE)
        .shuffle(1000)
        .batch(BATCH_SIZE)
        .prefetch(AUTOTUNE)
    )
    test_ds = (
        test_raw
        .map(preprocess, num_parallel_calls=AUTOTUNE)
        .batch(BATCH_SIZE)
        .prefetch(AUTOTUNE)
    )


    x_test_images = []
    for batch in test_ds.take(5):
        x_test_images.append(batch.numpy())
    x_test_images = np.concatenate(x_test_images, axis=0)

    return train_ds, test_ds, x_test_images



In [ ]:
class Sampling(layers.Layer):
    """
    Reparameterization trick: z = μ + exp(0.5 * log_var) * ε
    where ε ~ N(0, I).
    """
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon



In [ ]:
def build_encoder(latent_dim):
    """
    Encoder: (64x64x3) -> (z_mean, z_log_var, z)
    """
    encoder_inputs = keras.Input(shape=IMAGE_SHAPE)

    x = layers.Conv2D(32, 3, strides=2, padding="same", activation="relu")(encoder_inputs)   # 32x32
    x = layers.Conv2D(64, 3, strides=2, padding="same", activation="relu")(x)                # 16x16
    x = layers.Conv2D(128, 3, strides=2, padding="same", activation="relu")(x)               # 8x8
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)

    z_mean = layers.Dense(latent_dim, name="z_mean")(x)
    z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)
    z = Sampling()([z_mean, z_log_var])

    encoder = keras.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")
    return encoder


def build_decoder(latent_dim):
    """
    Decoder: z -> reconstructed face (64x64x3)
    """
    latent_inputs = keras.Input(shape=(latent_dim,))

    x = layers.Dense(8 * 8 * 128, activation="relu")(latent_inputs)
    x = layers.Reshape((8, 8, 128))(x)

    x = layers.Conv2DTranspose(128, 3, strides=2, padding="same", activation="relu")(x)  # 16x16
    x = layers.Conv2DTranspose(64, 3, strides=2, padding="same", activation="relu")(x)   # 32x32
    x = layers.Conv2DTranspose(32, 3, strides=2, padding="same", activation="relu")(x)   # 64x64

      decoder_outputs = layers.Conv2DTranspose(3, 3, padding="same", activation="sigmoid")(x)

    decoder = keras.Model(latent_inputs, decoder_outputs, name="decoder")
    return decoder



In [ ]:
class VAE(keras.Model):
    def __init__(self, encoder, decoder, kl_weight=1.0, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.kl_weight = kl_weight

        self.total_loss_tracker = keras.metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.reconstruction_loss_tracker,
            self.kl_loss_tracker,
        ]

    def call(self, inputs, training=False):
        z_mean, z_log_var, z = self.encoder(inputs)
        reconstruction = self.decoder(z)
        return reconstruction

    def train_step(self, data):
        if isinstance(data, tuple):
            data = data[0]

        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data, training=True)
            reconstruction = self.decoder(z, training=True)

            mse = tf.square(data - reconstruction)
            mse_per_example = tf.reduce_sum(mse, axis=(1, 2, 3))
            reconstruction_loss = tf.reduce_mean(mse_per_example)


            kl_loss = -0.5 * tf.reduce_sum(
                1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var),
                axis=1
            )
            kl_loss = tf.reduce_mean(kl_loss)

            total_loss = reconstruction_loss + self.kl_weight * kl_loss

        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)

        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }


In [ ]:
def show_face_reconstructions(vae, test_dataset, num_images=8):
    """
    Shows a few original vs reconstructed faces from the test dataset.
    """
    # Take one batch
    for batch in test_dataset.take(1):
        original_images = batch
        break

    original_images = original_images[:num_images]
    reconstructed_images = vae.predict(original_images)

    plt.figure(figsize=(2 * num_images, 4))

    for i in range(num_images):
        # Original
        ax = plt.subplot(2, num_images, i + 1)
        plt.imshow(original_images[i].numpy())
        ax.axis("off")
        if i == 0:
            ax.set_title("Original")

        # Reconstruction
        ax = plt.subplot(2, num_images, i + 1 + num_images)
        plt.imshow(reconstructed_images[i])
        ax.axis("off")
        if i == 0:
            ax.set_title("Reconstructed")

    plt.tight_layout()
    plt.show()


def sample_random_faces(decoder, num_samples=8):

    z_random = np.random.normal(size=(num_samples, LATENT_DIM))
    generated_faces = decoder.predict(z_random)

    plt.figure(figsize=(2 * num_samples, 2))
    for i in range(num_samples):
        ax = plt.subplot(1, num_samples, i + 1)
        plt.imshow(generated_faces[i])
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def run_vae_faces_lfw():
    print(f"TensorFlow Version: {tf.__version__}")
    print(f"Latent Dimensions (z): {LATENT_DIM}")
    print(f"Training for {EPOCHS} epochs with batch size {BATCH_SIZE}")
    print("Loading LFW faces from TensorFlow Datasets (tfds.load('lfw')) ...")

    train_dataset, test_dataset, x_test_images = load_and_preprocess_faces_from_lfw()

    encoder = build_encoder(LATENT_DIM)
    decoder = build_decoder(LATENT_DIM)

    vae = VAE(encoder, decoder, kl_weight=KL_WEIGHT)
    vae.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE))

    print("\nEncoder summary:")
    encoder.summary()
    print("\nDecoder summary:")
    decoder.summary()

    print("\n--- Starting VAE Training on LFW Faces ---")
    vae.fit(train_dataset, epochs=EPOCHS)
    print("--- Training Complete ---\n")

    print("\n--- Reconstruction Test on LFW Faces ---")
    show_face_reconstructions(vae, test_dataset, num_images=8)

    print("\n--- Sampling Random Faces from Latent Space ---")
    sample_random_faces(decoder, num_samples=8)

In [ ]:
if __name__ == "__main__":
    run_vae_faces_lfw()

TensorFlow Version: 2.19.0
Latent Dimensions (z): 32
Training for 20 epochs with batch size 64
Loading LFW faces from TensorFlow Datasets (tfds.load('lfw')) ...

Encoder summary:


Model: "encoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 64, 64, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 32, 32,    │        896 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 16, 16,    │     18,496 │ conv2d[0][0]      │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 8, 8, 128) │     73,856 │ conv2d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 8192)      │          0 │ conv2d_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │  2,097,408 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ z_mean (Dense)      │ (None, 32)        │      8,224 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ z_log_var (Dense)   │ (None, 32)        │      8,224 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sampling (Sampling) │ (None, 32)        │          0 │ z_mean[0][0],     │
│                     │                   │            │ z_log_var[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,207,104 (8.42 MB)

 Trainable params: 2,207,104 (8.42 MB)

...

 Total params: 511,043 (1.95 MB)

 Trainable params: 511,043 (1.95 MB)

 Non-trainable params: 0 (0.00 B)


--- Starting VAE Training on LFW Faces ---
Epoch 1/20
 20/193 ━━━━━━━━━━━━━━━━━━━━ 2:59 1s/step - kl_loss: 0.2314 - loss: 1119.1777 - reconstruction_loss: 1119.1772

KeyboardInterrupt: 